In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib as plt
plt.rcParams["figure.figsize"] = (10,6)
df = pd.read_csv(r"C:\Users\HP\OneDrive\Desktop\python\data\raw data\yellow_tripdata_2015-01.csv")
print("Shape : ",df.shape)
df.head()

In [ ]:
df.info()
df.dtypes

In [ ]:
missing = (
    df.isnull().sum().to_frame("Missing values")
)
missing["percentage"] = (missing["Missing values"]/len(df))*100
missing.sort_values(
    by= "percentage" , 
    ascending= True,
    inplace= False
)

In [ ]:
df.duplicated().sum()
duplicates = df[df.duplicated()]
duplicates.head()

In [ ]:
df.describe()

In [10]:
df["tpep_dropoff_datetime"] = pd.to_datetime(
    df["tpep_dropoff_datetime"],
    errors= "coerce"
)
df["tpep_pickup_datetime"] = pd.to_datetime(
    df["tpep_pickup_datetime"],
    errors= "coerce"
)

In [21]:
df["Trip_duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds()/60

In [ ]:
df.Trip_duration.describe()

In [ ]:
print("Minimum Duration:", df.Trip_duration.min(), "minutes")
print("Maximum Duration:", df.Trip_duration.max(), "minutes")
print("Average Duration:", round(df.Trip_duration.mean(), 2), "minutes")


In [ ]:
negative = df[df["Trip_duration"]<0]
print("Number of negative trips : ",len(negative))
negative.head(10)


In [ ]:
long_trips = df[df.Trip_duration > 600]
print(len(long_trips))
long_trips.head(10)

In [ ]:
total_rows = len(df)
percentage_negative = round((len(negative)/total_rows)*100 , 5)
print("The percentage of trips that are negative :" , percentage_negative)

In [ ]:
zero_distance = (df.trip_distance ==0).sum()
negative_distance = (df.trip_distance <0).sum()
print("Trips with negative distance :",negative_distance)
df[df["trip_distance"]<0].head(10)


In [ ]:
negative_fare = (df.total_amount <0).sum()
zero_fare = (df.total_amount ==0).sum()
print("Trips with zero Fare :" , zero_fare)
df[df["total_amount"]==0].head(10)

In [ ]:
zero_passengers = (df.passenger_count ==0).sum()
negative_passengers = (df.passenger_count <0).sum()
high_passengers = (df.passenger_count >8).sum()

print("No. of trips with more than 8 passengers :" , high_passengers)
df[df["passenger_count"]>8].head(10)

In [ ]:
df["average_speed_mph"] = np.where(
    df["Trip_duration"] > 0,
    df["trip_distance"] / (df["Trip_duration"] / 60),
    np.nan
)

In [ ]:
fast_trips = (df.average_speed_mph > 100).sum()
print("Trips with speed more than 100 mph :", fast_trips)
df[df["average_speed_mph"]>100].head()

In [ ]:
missing_coordinates = df[
    ["pickup_latitude",
     "pickup_longitude",
     "dropoff_latitude",
     "dropoff_longitude"]].isnull().sum()
print(missing_coordinates)


In [69]:
quality_report = pd.DataFrame({
    "Issue": [
        "Missing Values",
        "Duplicate Rows",
        "Negative Duration",
        "Zero Distance",
        "Negative Distance",
        "Zero Fare",
        "Negative Fare",
        "Passenger Count = 0",
        "Passenger Count > 8",
        "Trips > 100 mph",
        "Trips > 10 Hours",
        
    ],
    "Count": [
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        (df["Trip_duration"] < 0).sum(),
        (df["trip_distance"] == 0).sum(),
        (df["trip_distance"] < 0).sum(),
        (df["total_amount"] == 0).sum(),
        (df["total_amount"] < 0).sum(),
        (df["passenger_count"] == 0).sum(),
        (df["passenger_count"] > 8).sum(),
        (df["average_speed_mph"] > 100).sum(),
        (df["Trip_duration"] > 600).sum(),
    ]
})
quality_report

,Issue,Count
0,Missing Values,15116
1,Duplicate Rows,383
2,Negative Duration,297
3,Zero Distance,79365
4,Negative Distance,0
5,Zero Fare,1194
6,Negative Fare,4067
7,Passenger Count = 0,6565
8,Passenger Count > 8,11
9,Trips > 100 mph,8282
